# Day 22：前端速成：HTML/CSS/JS 基础、fetch 通信与企业聊天界面工程（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 4 周**：第4周：全栈 Web 聊天应用与 FastAPI 后端工程  
> **GitLab 工单**：`STARMIND-022` ｜ **开发分支**：`feat/day22-frontend-chat-ui`  
> **配套 HTML5 课件**：`courseware/day22/index.html` ｜ **独立版本快照**：`versions/day22/`

---

## 🎙️ 老陈开场旁白

早啊各位！欢迎进入第 4 周的全栈冲刺！很多做 AI 和 Python 后端的同学一听要写前端就头大，其实咱们做 AI 工程师不需要去卷复杂的前端框架底层原理，只要花一天时间吃透 **HTML 语义结构 + CSS Flex/Grid 布局 + JavaScript `fetch` 异步通信与 DOM 安全渲染**，就能徒手撸出媲美 ChatGPT 的企业级聊天工作台！今天老陈用工程化思路带大家：用 Python 生成一套完整的 `index.html` + `styles.css` + `chat_app.js` 静态站点资产，再写一个静态前端校验器，自动检查 HTML 标签闭合、防 XSS 注入（警惕 `innerHTML`）以及 `fetch` API 契约一致性！

### 承上启下
- **昨日基线**：昨天（Day 21）咱们以 100 分 A+ 通过了第 3 周综合周测，把多轮记忆、安全过滤、工具调用和流式事件引擎全部打磨完毕。但林产品发来灵魂拷问：公司 800 个非技术同事不可能打开终端敲 `python main.py` 来用星辰智脑，我们必须有一个打开浏览器就能聊天的 Web 界面！
- **明日预告**：明天（Day 23）我们正式进入 FastAPI 后端开发（上）：掌握路由、路径/查询参数、Pydantic v2 强类型校验，把 AI 对话封装成标准 REST API 并用 `TestClient` 自动化测试。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day22'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-022）

星辰智脑需要一套零重型构建依赖（无需 npm/webpack 也能直接被 FastAPI 托管）的企业级 Web 聊天前端，包含侧边会话栏、消息气泡区、工具状态徽章、输入框与 `fetch` 异步/流式接口对接逻辑。

**今日迭代目标：**
- 编写 `day22_frontend_builder.py`，生成结构清晰、暗亮色变量规范、响应式布局的企业聊天前端三件套（`index.html`、`styles.css`、`chat_app.js`）。
- 在 `chat_app.js` 中使用现代 `async/await` + `fetch('/api/v1/chat/stream', {method: 'POST'})` 与 `ReadableStream` 处理流式响应，且严格使用 `textContent` 防止 XSS。
- 编写 `day22_frontend_validator.py`，基于 Python 标准库 `html.parser` 自动校验标签闭合、DOM 必需 ID、API 端点契约与 XSS 安全规范。

**验收标准：**
- 一键生成的 `index.html`、`styles.css`、`chat_app.js` 全部落盘可读且字节非空。
- HTML 校验器验证所有非自闭合标签 100% 匹配闭合，包含聊天容器、表单、输入框和发送按钮关键元素。
- JS 安全校验器确认包含 `fetch` 调用、目标端点 `/api/v1/chat/stream` 且零不安全 `innerHTML` 赋值。

---

## 📚 课堂核心技术笔记

### 1. 1. HTML5 语义化标签与表单拦截

不要满屏只写 `<div>`，合理使用 `<aside>`、`<header>`、`<main>`、`<form>` 能让结构一目了然。监听表单提交事件 `form.addEventListener('submit', async (e) => { e.preventDefault(); ... })` 时，务必调用 `e.preventDefault()` 阻止浏览器默认的整页刷新跳转。

> **⚠️ 老陈避坑**：新手常忘记 `e.preventDefault()`，一点「发送」按钮页面就刷新重置，聊天记录瞬间清空。

### 2. 2. 为什么流式对话要用 `fetch` 而不是只用 `EventSource`？

浏览器原生的 `new EventSource(url)` 虽然能自动解析 SSE，但它**只支持 GET 请求**且不能自定义 `Authorization` 请求头！在企业场景中，对话消息很长且必须携带 JWT Token，因此生产标准做法是使用 `fetch(url, { method: 'POST', headers: {...}, body: JSON.stringify(...) })`，然后通过 `response.body.getReader()` + `new TextDecoder()` 手动按行解析 `data: ` 帧。

> **⚠️ 老陈避坑**：解析流式 chunk 时，单次网络包可能把一行 JSON 截成两半，必须维护一个字符串 `buffer`，按 `\n` 切分，把最后未闭合的半行留到下一次 `read()` 拼接！

### 3. 3. 前端安全第一铁律：防范 XSS 注入

永远优先使用 `node.textContent = content` 写入文本。如果确实需要渲染 Markdown 富文本，必须经过专业清洗库（如 DOMPurify）净化后再挂载，绝不能裸写 `div.innerHTML = userInput`。

> **⚠️ 老陈避坑**：如果用户发一句 `<img src=x onerror=alert(document.cookie)>` 而前端用了 `innerHTML`，立刻触发 XSS 窃取登录凭证！


## 💻 实操步骤 1：步骤一：用 Python 生成星辰智脑企业级静态聊天前端（HTML/CSS/JS）

- **目标文件**：`versions/day22/src/day22_frontend_builder.py`
- **老陈代码旁白**：咱们先写 `src/day22_frontend_builder.py`，一键生成颜值在线、交互完整、安全防 XSS 的星辰智脑静态聊天前端三件套（`index.html`、`styles.css`、`chat_app.js`），直接为 Day 24 的网页版 ChatGPT 克隆前后端联调备好弹药！


In [ ]:
"""
StarMind Day 22 - 步骤一：星辰智脑企业级静态聊天前端生成器
生成完整、无外部 CDN 依赖的 index.html、styles.css 与 chat_app.js（支持 fetch 流式读取）。
"""

from pathlib import Path
from typing import Dict


INDEX_HTML = """<!DOCTYPE html>
<html lang="zh-CN">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>星辰智脑 StarMind · 企业 AI 工作台</title>
  <link rel="stylesheet" href="styles.css" />
</head>
<body>
  <div class="app-shell">
    <aside class="sidebar">
      <div class="brand">
        <span class="brand-badge">SM</span>
        <div>
          <h1>星辰智脑</h1>
          <p>杭州星辰数科 AI 平台</p>
        </div>
      </div>
      <button id="new-chat-btn" class="primary-btn" type="button">+ 新建企业会话</button>
      <ul id="session-list" class="session-list">
        <li class="session-item active">当前会话 · 默认工作台</li>
      </ul>
    </aside>

    <main class="chat-main">
      <header class="chat-header">
        <div>
          <h2>智能问答与工具助手</h2>
          <span id="status-badge" class="status-badge">就绪 · SSE 流式通道</span>
        </div>
      </header>

      <section id="chat-log" class="chat-log" aria-live="polite">
        <div class="msg-row assistant">
          <div class="bubble">您好！我是星辰智脑企业 AI 助手，支持实时问答、差旅制度查询与流式打字机输出。</div>
        </div>
      </section>

      <form id="chat-form" class="chat-composer">
        <textarea id="chat-input" rows="2" placeholder="输入您的问题，例如：杭州差旅住宿报销上限是多少？" required></textarea>
        <button id="send-btn" class="send-btn" type="submit">发送</button>
      </form>
    </main>
  </div>
  <script src="chat_app.js"></script>
</body>
</html>
"""

STYLES_CSS = """:root {
  --bg-page: #f4f6fb;
  --bg-sidebar: #0f172a;
  --bg-panel: #ffffff;
  --accent: #2563eb;
  --accent-hover: #1d4ed8;
  --text-main: #1e293b;
  --text-muted: #64748b;
  --border: #e2e8f0;
}

* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: -apple-system, BlinkMacSystemFont, "PingFang SC", "Microsoft YaHei", sans-serif; background: var(--bg-page); color: var(--text-main); height: 100vh; }

.app-shell { display: flex; height: 100vh; }
.sidebar { width: 260px; background: var(--bg-sidebar); color: #f8fafc; padding: 20px; display: flex; flex-direction: column; gap: 16px; }
.brand { display: flex; align-items: center; gap: 12px; }
.brand-badge { width: 38px; height: 38px; border-radius: 10px; background: var(--accent); display: grid; place-items: center; font-weight: 700; }
.brand h1 { font-size: 18px; }
.brand p { font-size: 12px; color: #94a3b8; }

.primary-btn { background: var(--accent); color: #fff; border: none; border-radius: 8px; padding: 10px 14px; cursor: pointer; font-weight: 600; }
.primary-btn:hover { background: var(--accent-hover); }
.session-list { list-style: none; display: flex; flex-direction: column; gap: 8px; }
.session-item { padding: 10px 12px; border-radius: 8px; font-size: 13px; background: rgba(255, 255, 255, 0.08); }

.chat-main { flex: 1; display: flex; flex-direction: column; background: var(--bg-panel); }
.chat-header { padding: 16px 24px; border-bottom: 1px solid var(--border); display: flex; justify-content: space-between; align-items: center; }
.chat-header h2 { font-size: 16px; }
.status-badge { font-size: 12px; color: #059669; background: #ecfdf5; padding: 4px 10px; border-radius: 999px; }

.chat-log { flex: 1; overflow-y: auto; padding: 24px; display: flex; flex-direction: column; gap: 16px; }
.msg-row { display: flex; }
.msg-row.user { justify-content: flex-end; }
.msg-row.assistant { justify-content: flex-start; }
.bubble { max-width: 72%; padding: 12px 16px; border-radius: 12px; line-height: 1.6; font-size: 14px; white-space: pre-wrap; }
.msg-row.user .bubble { background: var(--accent); color: #ffffff; }
.msg-row.assistant .bubble { background: #f1f5f9; color: var(--text-main); }

.chat-composer { padding: 16px 24px; border-top: 1px solid var(--border); display: flex; gap: 12px; }
.chat-composer textarea { flex: 1; resize: none; padding: 10px 14px; border: 1px solid var(--border); border-radius: 10px; font-size: 14px; }
.send-btn { background: var(--accent); color: #fff; border: none; border-radius: 10px; padding: 0 22px; font-weight: 600; cursor: pointer; }
.send-btn:disabled { opacity: 0.6; cursor: not-allowed; }
"""

CHAT_APP_JS = """// 星辰智脑 StarMind 前端交互与流式通信脚本（严格使用 textContent 防 XSS）
const API_STREAM_ENDPOINT = "/api/v1/chat/stream";

let currentSessionId = "web_session_" + Date.now();

const chatLog = document.getElementById("chat-log");
const chatForm = document.getElementById("chat-form");
const chatInput = document.getElementById("chat-input");
const sendBtn = document.getElementById("send-btn");
const statusBadge = document.getElementById("status-badge");
const newChatBtn = document.getElementById("new-chat-btn");

function appendMessageBubble(role, initialText) {
  const row = document.createElement("div");
  row.className = "msg-row " + role;
  const bubble = document.createElement("div");
  bubble.className = "bubble";
  bubble.textContent = initialText;
  row.appendChild(bubble);
  chatLog.appendChild(row);
  chatLog.scrollTop = chatLog.scrollHeight;
  return bubble;
}

async function sendStreamingMessage(userText) {
  sendBtn.disabled = true;
  statusBadge.textContent = "思考与流式输出中...";
  appendMessageBubble("user", userText);
  const aiBubble = appendMessageBubble("assistant", "");

  try {
    const response = await fetch(API_STREAM_ENDPOINT, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ session_id: currentSessionId, message: userText }),
    });
    if (!response.ok || !response.body) {
      throw new Error("HTTP " + response.status);
    }
    const reader = response.body.getReader();
    const decoder = new TextDecoder("utf-8");
    let buffer = "";

    while (true) {
      const { value, done } = await reader.read();
      if (done) break;
      buffer += decoder.decode(value, { stream: true });
      const lines = buffer.split("\\n");
      buffer = lines.pop() || "";
      for (const line of lines) {
        if (line.startsWith("data: ")) {
          const raw = line.slice(6).trim();
          if (!raw || raw === "[DONE]") continue;
          const payload = JSON.parse(raw);
          if (payload.delta) {
            aiBubble.textContent += payload.delta;
            chatLog.scrollTop = chatLog.scrollHeight;
          }
        }
      }
    }
    statusBadge.textContent = "就绪 · SSE 流式通道";
  } catch (err) {
    aiBubble.textContent = "请求异常，请确认 FastAPI 服务已启动：" + err.message;
    statusBadge.textContent = "连接异常";
  } finally {
    sendBtn.disabled = false;
  }
}

if (chatForm) {
  chatForm.addEventListener("submit", async (event) => {
    event.preventDefault();
    const text = chatInput.value.trim();
    if (!text) return;
    chatInput.value = "";
    await sendStreamingMessage(text);
  });
}

if (newChatBtn) {
  newChatBtn.addEventListener("click", () => {
    currentSessionId = "web_session_" + Date.now();
    while (chatLog.firstChild) {
      chatLog.removeChild(chatLog.firstChild);
    }
    appendMessageBubble("assistant", "已开启全新企业会话，请问有什么可以帮您？");
  });
}
"""


def build_frontend_assets(output_dir: Path) -> Dict[str, Path]:
    """将完整的企业聊天前端三件套写入目标目录并返回路径字典。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    files = {
        "html": output_dir / "index.html",
        "css": output_dir / "styles.css",
        "js": output_dir / "chat_app.js",
    }
    files["html"].write_text(INDEX_HTML.strip() + "\n", encoding="utf-8")
    files["css"].write_text(STYLES_CSS.strip() + "\n", encoding="utf-8")
    files["js"].write_text(CHAT_APP_JS.strip() + "\n", encoding="utf-8")
    return files


if __name__ == "__main__":
    out = Path("generated_web_demo")
    created = build_frontend_assets(out)
    for kind, path in created.items():
        print(f"已生成 [{kind.upper()}]: {path} ({path.stat().st_size} bytes)")


### 🔍 步骤 1 代码深度解析

1. `INDEX_HTML` 严格使用语义化标签（`<aside>`、`<main>`、`<header>`、`<section>`、`<form>`）并声明 `aria-live='polite'` 无障碍属性。
2. `CHAT_APP_JS` 全程使用 `textContent` 与 `createElement`，绝无 `innerHTML` 隐患，并且完整实现了 `fetch` + `ReadableStream` + `buffer` 拆行防粘包逻辑。
3. `build_frontend_assets(output_dir)` 方便后续被 FastAPI 或单元测试一键生成到任意目录。

**预期运行输出参考：**
```text
已生成 [HTML]: generated_web_demo/index.html (... bytes)
已生成 [CSS]: generated_web_demo/styles.css (... bytes)
已生成 [JS]: generated_web_demo/chat_app.js (... bytes)
```


## 💻 实操步骤 2：步骤二：实现静态前端 DOM 结构、CSS 规范与 JS 安全自动校验器

- **目标文件**：`versions/day22/src/day22_frontend_validator.py`
- **老陈代码旁白**：前端代码不能光靠肉眼看，咱们写 `src/day22_frontend_validator.py`！它用 Python 标准库 `html.parser` 检查标签闭合平衡和核心 DOM ID，用正则审计 JS 是否遵守 `textContent` 防 XSS 规范以及 `fetch` 端点是否与后端契约一致。


In [ ]:
"""
StarMind Day 22 - 步骤二：企业前端静态资产自动校验器
校验 HTML 标签闭合平衡、必需交互元素 ID、CSS 主题变量以及 JS 防 XSS 与 API 端点契约。
"""

import re
from dataclasses import dataclass, field
from html.parser import HTMLParser
from pathlib import Path
from typing import Dict, List, Set


VOID_ELEMENTS = {"meta", "link", "br", "hr", "img", "input"}


class _HTMLStructureParser(HTMLParser):
    """基于标准库 HTMLParser 检查标签闭合栈与收集全部元素 ID。"""

    def __init__(self) -> None:
        super().__init__()
        self.stack: List[str] = []
        self.ids: Set[str] = set()
        self.errors: List[str] = []

    def handle_starttag(self, tag: str, attrs: List[tuple[str, str | None]]) -> None:
        for k, v in attrs:
            if k == "id" and v:
                self.ids.add(v)
        if tag not in VOID_ELEMENTS:
            self.stack.append(tag)

    def handle_endtag(self, tag: str) -> None:
        if tag in VOID_ELEMENTS:
            return
        if not self.stack:
            self.errors.append(f"多余的闭合标签 </{tag}>")
            return
        last = self.stack.pop()
        if last != tag:
            self.errors.append(f"标签嵌套不匹配：期望 </{last}>，实际遇到 </{tag}>")


@dataclass
class ValidationReport:
    passed: bool
    found_ids: Set[str] = field(default_factory=set)
    errors: List[str] = field(default_factory=list)


def validate_frontend_bundle(
    asset_dir: Path,
    required_ids: Set[str] | None = None,
    expected_api_endpoint: str = "/api/v1/chat/stream",
) -> ValidationReport:
    """对生成的静态前端目录执行端到端工程质量与安全校验。"""
    req_ids = required_ids or {"chat-log", "chat-form", "chat-input", "send-btn", "status-badge"}
    errors: List[str] = []

    html_path = asset_dir / "index.html"
    css_path = asset_dir / "styles.css"
    js_path = asset_dir / "chat_app.js"

    for p in (html_path, css_path, js_path):
        if not p.exists():
            errors.append(f"缺失静态资产文件: {p.name}")
    if errors:
        return ValidationReport(passed=False, errors=errors)

    # 1. HTML 结构与 DOM ID 校验
    html_text = html_path.read_text(encoding="utf-8")
    parser = _HTMLStructureParser()
    parser.feed(html_text)
    errors.extend(parser.errors)
    if parser.stack:
        errors.append(f"存在未闭合的 HTML 标签: {parser.stack}")
    missing_ids = req_ids - parser.ids
    if missing_ids:
        errors.append(f"HTML 缺少必需交互元素 ID: {sorted(missing_ids)}")

    # 2. CSS 主题变量规范校验
    css_text = css_path.read_text(encoding="utf-8")
    if ":root" not in css_text or "--accent" not in css_text:
        errors.append("CSS 未定义 :root 主题变量（至少应包含 --accent）")

    # 3. JS fetch 契约与 XSS 安全审计
    js_text = js_path.read_text(encoding="utf-8")
    if "fetch(" not in js_text:
        errors.append("JS 中未检测到 fetch() 异步通信调用")
    if expected_api_endpoint not in js_text:
        errors.append(f"JS 未对接约定的后端接口端点: {expected_api_endpoint}")
    if re.search(r"\.innerHTML\s*=", js_text):
        errors.append("安全告警：检测到不安全的 .innerHTML 赋值，请改用 .textContent 防范 XSS")

    return ValidationReport(passed=len(errors) == 0, found_ids=parser.ids, errors=errors)


if __name__ == "__main__":
    from tempfile import TemporaryDirectory
    from src.day22_frontend_builder import build_frontend_assets

    with TemporaryDirectory() as tmp:
        build_frontend_assets(Path(tmp))
        rep = validate_frontend_bundle(Path(tmp))
        print("前端资产校验结果:", rep)


### 🔍 步骤 2 代码深度解析

1. `_HTMLStructureParser` 用栈（Stack）数据结构验证 HTML 标签嵌套是否严格匹配，并忽略 `meta/link/input` 等自闭合空元素。
2. JS 安全规则主动扫描 `.innerHTML =` 赋值，把 XSS 隐患扼杀在提交代码之前。
3. 端点契约检查确保前端 `chat_app.js` 调用的路径与 Day 24 FastAPI 路由 100% 对齐。

**预期运行输出参考：**
```text
前端资产校验结果: ValidationReport(passed=True, found_ids={'chat-form', 'chat-input', 'chat-log', 'new-chat-btn', 'send-btn', 'session-list', 'status-badge'}, errors=[])
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day22/tests/test_day22.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 22 自动化单元测试：验证企业聊天前端生成、HTML闭合平衡与XSS安全校验。"""

from pathlib import Path
from src.day22_frontend_builder import build_frontend_assets
from src.day22_frontend_validator import validate_frontend_bundle


def test_build_and_validate_enterprise_frontend(tmp_path: Path):
    assets = build_frontend_assets(tmp_path)
    assert assets["html"].exists()
    assert assets["css"].exists()
    assert assets["js"].exists()

    report = validate_frontend_bundle(tmp_path)
    assert report.passed is True
    assert report.errors == []
    assert {"chat-log", "chat-form", "chat-input", "send-btn"}.issubset(report.found_ids)


def test_validator_catches_xss_and_unclosed_html(tmp_path: Path):
    build_frontend_assets(tmp_path)
    # 故意注入不安全的 innerHTML 赋值与未闭合标签
    (tmp_path / "chat_app.js").write_text("fetch('/api/v1/chat/stream'); el.innerHTML = userText;", encoding="utf-8")
    (tmp_path / "index.html").write_text("<html><body><div id='chat-log'>未闭合</body></html>", encoding="utf-8")

    report = validate_frontend_bundle(tmp_path)
    assert report.passed is False
    assert any("innerHTML" in err for err in report.errors)
    assert any("标签" in err for err in report.errors)

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为聊天前端新增暗黑模式（Dark Theme）CSS 主题变量块

**题目要求：**
编写函数 `append_dark_theme(css_text)`，在现有 CSS 末尾追加 `[data-theme="dark"]` 变量覆盖块（覆盖 `--bg-page`、`--bg-panel`、`--text-main`、`--border`），并验证输出包含暗黑主题选择器。

**思路提示：** 返回拼接后的 CSS 字符串，确保包含 `[data-theme="dark"]` 与四大核心变量。


In [ ]:
def append_dark_theme(css_text: str) -> str:
    """为企业聊天前端追加暗黑模式 CSS 变量定义。"""
    dark_block = """
[data-theme="dark"] {
  --bg-page: #0b1120;
  --bg-panel: #111827;
  --text-main: #f1f5f9;
  --border: #1f2937;
}
"""
    return css_text.rstrip() + "\n" + dark_block


if __name__ == "__main__":
    from src.day22_frontend_builder import STYLES_CSS
    updated = append_dark_theme(STYLES_CSS)
    print("包含暗黑模式:", '[data-theme="dark"]' in updated)


**答案解析与评分标准：**

借助 CSS `:root` 自定义属性，只需在 `document.documentElement.dataset.theme = 'dark'` 切换属性即可实现零闪烁换肤。


### 作业 2（企业实战）：轻量级纯 Python HTML 转义工具函数（防 XSS）

**题目要求：**
编写 `safe_render_bubble_html(role, content)`，使用标准库 `html.escape` 对 `content` 执行安全转义后，再拼装成 `<div class="msg-row {role}"><div class="bubble">...</div></div>`，验证 `<script>alert(1)</script>` 被安全转义。

**思路提示：** 调用 `html.escape(content, quote=True)`，同时校验 `role in {'user', 'assistant'}`。


In [ ]:
import html


def safe_render_bubble_html(role: str, content: str) -> str:
    """服务端安全渲染消息气泡 HTML，严格防 XSS 注入。"""
    if role not in {"user", "assistant"}:
        raise ValueError(f"非法角色: {role}")
    escaped = html.escape(content, quote=True)
    return f'<div class="msg-row {role}"><div class="bubble">{escaped}</div></div>'


if __name__ == "__main__":
    print(safe_render_bubble_html("user", "<script>alert('xss')</script>"))


**答案解析与评分标准：**

服务端渲染（SSR）与客户端渲染（CSR）都必须遵循「不信任一切外部文本」的转义原则。


### 作业 3（架构进阶）：SSE 跨网络包粘包/拆包解析器（Python 模拟前端缓冲逻辑）

**题目要求：**
网络传输中 SSE 数据可能被任意字节边界切碎（例如 `['data: {"del', 'ta":"你"}\n\ndata: {"delta":"好"}\n\n']`）。请编写 `parse_chunked_sse(raw_chunks)` 还原所有完整 JSON 对象列表。

**思路提示：** 维护 `buffer = ''`，每次追加 chunk 后用 `buffer.split('\n')`，把最后一项留作新的 `buffer`，解析所有以 `data: ` 开头的完整行。


In [ ]:
import json
from typing import Any, Dict, Iterable, List


def parse_chunked_sse(raw_chunks: Iterable[str]) -> List[Dict[str, Any]]:
    """模拟前端 ReadableStream 缓冲机制，正确处理任意切分的 SSE 文本流。"""
    results: List[Dict[str, Any]] = []
    buffer = ""
    for chunk in raw_chunks:
        buffer += chunk
        lines = buffer.split("\n")
        buffer = lines.pop()
        for line in lines:
            if line.startswith("data: "):
                payload = line[6:].strip()
                if payload and payload != "[DONE]":
                    results.append(json.loads(payload))
    return results


if __name__ == "__main__":
    messy_chunks = ['data: {"del', 'ta":"你"}\n\ndata: {"del', 'ta":"好"}\n\n']
    print("解析结果:", parse_chunked_sse(messy_chunks))


**答案解析与评分标准：**

理解 TCP 字节流「无消息边界」的本质，是写好任何流式前后端解析器的基本功。


---

## 🎯 老陈收尾复盘

今天大家不仅速通了 HTML/CSS/JS 和 `fetch` 流式读取，还用 Python 工程化手段实现了前端资产生成与自动化安全巡检！记住一条血泪教训：渲染大模型返回的文本时，千万不要图省事直接把用户输入或未转义内容塞进 `element.innerHTML`，一定要用 `textContent` 防 XSS 跨站脚本攻击！明天咱们用 FastAPI 把后端接口架起来！
